# Build Water Infrastructure Dictionary

Builds `output/water_infrastructure_dictionary.csv` — major CA water infrastructure features for use as a spaCy entity ruler dictionary in the IRWM NER pipeline.

Each row has:
- `all_names`: pipe-separated list of the primary name + aliases (the R pipeline splits on `|`)
- `infra_type`: infrastructure category
- `operator`: owning/operating agency
- `notes`

**Sources / update guidance:**
- CA DWR State Water Project: https://water.ca.gov/Programs/State-Water-Project
- Bureau of Reclamation Central Valley Project: https://www.usbr.gov/mp/cvp/
- DWR dam inventory: https://water.ca.gov/Programs/Dam-Safety
- National Inventory of Dams (Army Corps): https://nid.usace.army.mil/

In [1]:
import os
import pandas as pd

OUTPUT_DIR = "../../output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "water_infrastructure_dictionary.csv")

## Major Dams

In [2]:
dams = [
    # State Water Project dams
    ("Oroville Dam|Lake Oroville Dam", "dam", "DWR", "tallest dam in US; on Feather River"),
    ("San Luis Dam|San Luis Reservoir Dam", "dam", "DWR/USBR", "off-stream storage; joint SWP/CVP"),
    ("Castaic Dam|Castaic Lake Dam", "dam", "DWR", "SWP terminal reservoir"),
    ("Perris Dam|Lake Perris Dam", "dam", "DWR", "SWP terminal reservoir"),
    ("Pyramid Dam|Pyramid Lake Dam", "dam", "DWR", "SWP on Piru Creek"),
    ("Silverwood Lake Dam|Cedar Springs Dam", "dam", "DWR", "SWP in San Bernardino County"),
    ("Thermalito Diversion Dam", "dam", "DWR", "on Feather River"),
    # CVP dams (Bureau of Reclamation)
    ("Shasta Dam|Shasta Lake Dam", "dam", "USBR", "CVP; on Sacramento River"),
    ("Keswick Dam", "dam", "USBR", "CVP re-regulation dam below Shasta"),
    ("Folsom Dam|Folsom Lake Dam", "dam", "USBR/Army Corps", "CVP; on American River"),
    ("Nimbus Dam", "dam", "USBR", "CVP re-regulation dam below Folsom"),
    ("Trinity Dam|Clair Engle Lake Dam", "dam", "USBR", "CVP; trans-basin diversion to Sacramento"),
    ("Whiskeytown Dam|Whiskeytown Lake Dam", "dam", "USBR", "CVP"),
    ("New Melones Dam|New Melones Lake Dam", "dam", "Army Corps/USBR", "CVP; on Stanislaus River"),
    ("Friant Dam|Millerton Lake Dam", "dam", "USBR", "CVP; on San Joaquin River"),
    ("Pine Flat Dam|Pine Flat Lake Dam", "dam", "Army Corps", "on Kings River"),
    ("Terminus Dam|Lake Kaweah Dam", "dam", "Army Corps", "on Kaweah River"),
    ("Success Dam|Lake Success Dam", "dam", "Army Corps", "on Tule River"),
    ("Isabella Dam|Lake Isabella Dam", "dam", "Army Corps", "on Kern River"),
    # Local agency dams
    ("Hetch Hetchy Dam|O'Shaughnessy Dam", "dam", "SFPUC", "on Tuolumne River in Yosemite"),
    ("New Don Pedro Dam|Don Pedro Reservoir Dam", "dam", "TID/MID", "on Tuolumne River"),
    ("New Exchequer Dam|Lake McClure Dam", "dam", "MID", "on Merced River"),
    ("New Bullards Bar Dam|Englebright Lake Dam", "dam", "NID", "on Yuba River"),
    ("Pardee Dam|Pardee Reservoir Dam", "dam", "EBMUD", "on Mokelumne River"),
    ("Camanche Dam|Camanche Reservoir Dam", "dam", "EBMUD", "on Mokelumne River"),
    ("Calaveras Dam", "dam", "SFPUC", "on Calaveras Creek"),
    ("Crystal Springs Dam|Lower Crystal Springs Reservoir", "dam", "SFPUC", ""),
    ("Anderson Dam|Anderson Reservoir Dam", "dam", "Santa Clara Valley Water", "on Coyote Creek"),
    ("Chesbro Dam|Chesbro Reservoir", "dam", "Santa Clara Valley Water", ""),
    ("Coyote Dam|Coyote Reservoir Dam", "dam", "Santa Clara Valley Water", ""),
    ("Lexington Dam|Lexington Reservoir Dam", "dam", "Santa Clara Valley Water", ""),
    ("Cachuma Dam|Lake Cachuma Dam|Bradbury Dam", "dam", "USBR/COMB", "on Santa Ynez River"),
    ("Gibraltar Dam|Gibraltar Reservoir", "dam", "City of Santa Barbara", "on Santa Ynez River"),
    ("Diamond Valley Lake Dam|Eastside Reservoir Dam", "dam", "MWD", "off-stream storage"),
    ("Hoover Dam|Boulder Dam", "dam", "USBR", "on Colorado River; Nevada/Arizona border"),
    ("Parker Dam", "dam", "USBR", "on Colorado River; creates Lake Havasu"),
    ("Prado Dam|Prado Flood Control Basin", "dam", "Army Corps", "on Santa Ana River"),
    ("Warm Springs Dam|Lake Sonoma Dam", "dam", "Army Corps/SCWA", "on Dry Creek, Sonoma County"),
    ("Coyote Valley Dam|Lake Mendocino Dam", "dam", "Army Corps", "on East Fork Russian River"),
    ("Monticello Dam|Lake Berryessa Dam", "dam", "USBR", "on Putah Creek"),
    ("Stony Gorge Dam|East Park Dam", "dam", "USBR", "on Stony Creek"),
    ("Black Butte Dam", "dam", "Army Corps", "on Stony Creek"),
    ("Englebright Dam", "dam", "Army Corps", "on Yuba River"),
    ("Buchanan Dam|H.V. Eastman Lake", "dam", "Army Corps", "on Chowchilla River"),
    ("Hensley Dam|Hensley Lake", "dam", "Army Corps", "on Fresno River"),
]

## Canals and Conveyance

In [3]:
canals = [
    # State Water Project
    ("California Aqueduct|SWP California Aqueduct", "canal", "DWR", "main SWP conveyance, Sacramento-San Diego"),
    ("North Bay Aqueduct|NBA", "canal", "DWR", "SWP branch to Napa/Solano counties"),
    ("South Bay Aqueduct|SBA", "canal", "DWR", "SWP branch to Alameda/Santa Clara counties"),
    ("Coastal Branch Aqueduct|Coastal Branch", "canal", "DWR", "SWP branch to San Luis Obispo/Santa Barbara"),
    ("East Branch Extension", "canal", "DWR", "SWP extension in San Bernardino County"),
    # CVP
    ("Delta-Mendota Canal|DMC", "canal", "USBR", "CVP west-side delivery"),
    ("Friant-Kern Canal|FKC", "canal", "USBR", "CVP delivery from Friant Dam south"),
    ("Madera Canal", "canal", "USBR", "CVP delivery from Friant Dam north"),
    ("Tehama-Colusa Canal|TC Canal", "canal", "USBR", "CVP north-valley delivery"),
    ("Corning Canal", "canal", "USBR", "CVP branch off Tehama-Colusa Canal"),
    ("Sacramento Canals Unit", "canal", "USBR", "CVP"),
    ("Cross Valley Canal", "canal", "KCWA", "connects CVP to Kern County"),
    # Other major canals
    ("Los Angeles Aqueduct|LAA|First Los Angeles Aqueduct|Second Los Angeles Aqueduct", "canal", "LADWP", "Owens Valley to Los Angeles"),
    ("Colorado River Aqueduct|CRA", "canal", "MWD", "Colorado River to Southern California"),
    ("Hetch Hetchy Aqueduct", "canal", "SFPUC", "Tuolumne River to San Francisco"),
    ("Mokelumne Aqueduct", "canal", "EBMUD", "Mokelumne River to East Bay"),
    ("Tuolumne River Regional Water System", "canal", "TID/MID", ""),
    ("All-American Canal", "canal", "USBR/IID", "Colorado River to Imperial Valley"),
    ("Coachella Canal", "canal", "USBR/CVWD", "Colorado River to Coachella Valley"),
    ("Kern River Canal", "canal", "Kern County agencies", ""),
    ("Kings River Canal", "canal", "KRCD", ""),
    ("Redwood Valley County Water District Pipeline", "canal", "Redwood Valley CWD", ""),
]

## Pumping Plants and Facilities

In [4]:
pumping = [
    ("Harvey O. Banks Pumping Plant|Banks Pumping Plant|SWP Delta Pumping Plant", "pumping_plant", "DWR", "SWP Delta export"),
    ("C.W. Bill Jones Pumping Plant|Tracy Pumping Plant|CVP Delta Pumping Plant", "pumping_plant", "USBR", "CVP Delta export; formerly Tracy"),
    ("Clifton Court Forebay", "reservoir", "DWR", "SWP pre-pumping holding reservoir"),
    ("Edmonston Pumping Plant", "pumping_plant", "DWR", "SWP; highest single-lift pump in world"),
    ("Devil Canyon Powerplant", "pumping_plant", "DWR", "SWP"),
    ("Dos Amigos Pumping Plant", "pumping_plant", "USBR", "CVP"),
    ("O'Neill Forebay", "reservoir", "DWR/USBR", "joint SWP/CVP forebay at San Luis"),
]

## Water Treatment and Wastewater Facilities

In [5]:
treatment = [
    ("Regional Water Quality Control Plant|Palo Alto Regional Water Quality Control Plant", "wastewater_treatment", "City of Palo Alto", ""),
    ("Sacramento Regional County Sanitation District|Sacramento Regional Wastewater Treatment Plant|SRCSD", "wastewater_treatment", "SRCSD", ""),
    ("City of Los Angeles Hyperion Water Reclamation Plant|Hyperion Treatment Plant", "wastewater_treatment", "LADWP/LA Sanitation", ""),
    ("Donald C. Tillman Water Reclamation Plant|Tillman Water Reclamation Plant", "wastewater_treatment", "LA Sanitation", ""),
    ("Terminal Island Water Reclamation Plant", "wastewater_treatment", "LA Sanitation", ""),
    ("San Jose-Santa Clara Regional Wastewater Facility", "wastewater_treatment", "City of San Jose", ""),
    ("East Bay Dischargers Authority|EBDA", "wastewater_treatment", "EBDA", ""),
    ("Central Contra Costa Sanitary District|Central San", "wastewater_treatment", "Central San", ""),
    ("Dublin San Ramon Services District|DSRSD", "wastewater_treatment", "DSRSD", ""),
    ("Los Coyotes Water Reclamation Plant", "water_treatment", "LA County", ""),
    ("Metropolitan Water District Weymouth Treatment Plant|F.E. Weymouth Water Treatment Plant", "water_treatment", "MWD", ""),
    ("Metropolitan Water District Skinner Treatment Plant|Robert B. Diemer Treatment Plant", "water_treatment", "MWD", ""),
    ("Freeport Regional Water Authority|FRWA", "water_treatment", "FRWA", "Sacramento region"),
    ("R.C. Dillon Water Treatment Plant", "water_treatment", "EBMUD", ""),
    ("Mokelumne Aqueduct Water Treatment Plant", "water_treatment", "EBMUD", ""),
    ("Silicon Valley Advanced Water Purification Center|SVAWPC", "water_treatment", "Santa Clara Valley Water/SBVWSD", "indirect potable reuse"),
    ("Orange County Water District Groundwater Replenishment System|GWRS", "water_treatment", "OCWD", "largest water purification system for potable reuse"),
    ("West Basin Water Recycling Program", "water_treatment", "West Basin MWD", ""),
]

## Flood Control Infrastructure

In [6]:
flood_control = [
    ("Sacramento Weir|Sacramento Bypass Weir", "flood_control", "Army Corps/State", "flood bypass"),
    ("Fremont Weir", "flood_control", "Army Corps/State", "flood bypass into Yolo Bypass"),
    ("Yolo Bypass", "flood_control", "DWR/YCFCWCD", "major flood bypass west of Sacramento"),
    ("Sutter Bypass", "flood_control", "State", "Sacramento Valley flood bypass"),
    ("Sacramento Flood Control Project", "flood_control", "Army Corps", ""),
    ("American River Flood Control Project", "flood_control", "Army Corps/SAFCA", ""),
    ("Folsom Dam Modification Project", "flood_control", "Army Corps/USBR", "raises spillway capacity"),
    ("Natomas Levee Improvement Program|NLIP", "flood_control", "SAFCA", ""),
    ("Santa Ana River Mainstem Project", "flood_control", "Army Corps", ""),
    ("Prado Basin Sediment Management", "flood_control", "Army Corps", ""),
    ("Los Angeles River Revitalization", "flood_control", "Army Corps/City of LA", ""),
    ("San Gabriel River Watershed", "flood_control", "Army Corps/LA County", ""),
    ("Coyote Creek Flood Bypass", "flood_control", "Santa Clara Valley Water", ""),
]

## Compile and Write CSV

In [7]:
all_entries = dams + canals + pumping + treatment + flood_control

df = pd.DataFrame(all_entries, columns=["all_names", "infra_type", "operator", "notes"])

assert df["all_names"].str.startswith("|").sum() == 0
assert df["all_names"].str.endswith("|").sum() == 0
assert (df["all_names"].str.len() > 0).all()

df.to_csv(OUTPUT_PATH, index=False)
print(f"Written {len(df)} infrastructure entries to {OUTPUT_PATH}")
print(df["infra_type"].value_counts().to_string())

Written 105 infrastructure entries to ../../output/water_infrastructure_dictionary.csv
infra_type
dam                     45
canal                   22
flood_control           13
wastewater_treatment     9
water_treatment          9
pumping_plant            5
reservoir                2
